# DuckDB + Iceberg + MinIO

This notebook runs inside the `jupyter` Docker Compose service. It connects DuckDB to the Iceberg REST catalog and MinIO object storage.

In [2]:
import sys
import os

# Replace with the actual path to the folder containing your 'local' file
module_path = os.path.abspath("/app") 

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
from app.lakehouse import connect_lakehouse

con = connect_lakehouse()

In [4]:

con.sql("CREATE SCHEMA IF NOT EXISTS lakehouse_db.demo;")
con.sql("""
CREATE TABLE IF NOT EXISTS lakehouse_db.demo.notebook_orders (
    order_id INTEGER,
    customer VARCHAR,
    amount DOUBLE,
    status VARCHAR
)
WITH ('format-version' = '2');
""")
con.sql("DELETE FROM lakehouse_db.demo.notebook_orders;")
con.sql("""
INSERT INTO lakehouse_db.demo.notebook_orders VALUES
    (1, 'Ada', 125.50, 'paid'),
    (2, 'Linus', 89.90, 'paid'),
    (3, 'Grace', 42.00, 'pending');
""")

CountBIGINT
3


In [5]:
con.sql("""
SELECT
    status,
    count(*) AS orders,
    round(sum(amount), 2) AS total_amount
FROM lakehouse_db.demo.notebook_orders
GROUP BY status
ORDER BY status;
""").df()

,status,orders,total_amount
0,paid,2,215.4
1,pending,1,42.0


In [6]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
con.sql(
        f"""
        INSERT INTO lakehouse_db.demo.orders VALUES
            (5, 'Trino', TIMESTAMP '{now}', 5.50, 'paid'),
            (6, 'elis', TIMESTAMP '{now}', 12.90, 'paid')
        """
    )

CountBIGINT
2


In [7]:
con.sql("""
    SELECT *
    FROM lakehouse_db.demo.orders
""").df()

,order_id,customer,order_ts,amount,status
0,1,Ada,2026-11-14 22:35:45,125.5,paid
1,2,Linus,2026-11-14 22:35:45,89.9,paid
2,3,Grace,2026-11-14 22:35:45,42.0,pending
3,4,Marta,2026-11-14 22:35:45,240.1,paid
4,5,Trino,2026-11-14 22:37:05,5.5,paid
5,6,elis,2026-11-14 22:37:05,12.9,paid


In [8]:
con.sql("""
    SELECT snapshot_id, sequence_number, timestamp_ms
    FROM iceberg_snapshots(lakehouse_db.demo.orders);
""").df()

,snapshot_id,sequence_number,timestamp_ms
0,5368849104691546587,2,2026-11-14 22:37:05.377
1,5272809482929258775,1,2026-11-14 22:35:45.179


In [9]:
import os
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    region_name=os.environ.get("AWS_REGION", "us-east-1"),
)

response = s3.list_objects_v2(Bucket=os.environ.get("ICEBERG_WAREHOUSE", "warehouse"), MaxKeys=10)
[obj["Key"] for obj in response.get("Contents", [])]

['demo/notebook_orders/data/019e992c-cc46-795d-afae-d2e610aa4da7.parquet',
 'demo/notebook_orders/data/01a1dc34-cdc8-7e4b-a21a-79ca60ab146d.parquet',
 'demo/notebook_orders/metadata/00000-12c50357-c3e1-47ad-a5e6-b4f103a28669.metadata.json',
 'demo/notebook_orders/metadata/00000-ccff9371-6218-4e04-9a80-892a286db5dd.metadata.json',
 'demo/notebook_orders/metadata/00001-463e5f44-8c46-4f8e-a856-6678e22040f4.metadata.json',
 'demo/notebook_orders/metadata/00001-cf9e4c47-fedf-4b1f-846c-67de757f8c93.metadata.json',
 'demo/notebook_orders/metadata/7e1f8e6f-952c-465e-87e6-47f5a647f8ad-m0.avro',
 'demo/notebook_orders/metadata/a3337cf9-7765-4315-a57b-93d8e999c3a9-m0.avro',
 'demo/notebook_orders/metadata/snap-5870662531236250598-7298b3a1-ecde-4915-9001-c3fc5662d0da.avro',
 'demo/notebook_orders/metadata/snap-8745925286433801996-39684e48-fa99-4552-8505-092a55b32ab8.avro']